# `unplug_charger` FM/DiT Experiment Console

这份 notebook 现在是项目的正式实验总控入口。

它只做三件事：
- 选择实验配置和模块组合
- 调用训练 / audit / rollout 的 `.py` 入口
- 汇总 `summary.json` / `audit_report.json` / 本地评估结果

不要再把模型实现直接写在 notebook 里；真正的 source of truth 在 `src/`。


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys

CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for cand in CANDIDATES:
    if (cand / 'pyproject.toml').exists() and (cand / 'scripts' / 'run_autoresearch_trial.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root. Please open the notebook from the repo root or its notebooks/ directory.')

TRIAL_SCRIPT = PROJECT_ROOT / 'scripts' / 'run_autoresearch_trial.py'
DEFAULT_CONFIG = PROJECT_ROOT / 'configs' / 'fm_autodl_lab.json'


In [ ]:
# User choices
CONFIG_PATH = DEFAULT_CONFIG
STRATEGY = 'fm'
OBS_MODE = 'pcd'
ENCODER_NAME = 'pointnet_token'
BACKBONE_NAME = 'dit'
BASE_RUN_NAME = 'unplug_charger_transformer_fm_obs3_dit_v1_retrain_noamp_v1'
EXPERIMENT_NAME = 'notebook_baseline_500'
STAGE_EPOCHS = 500
CHECKPOINT_EVERY = 100
DEVICE = None  # e.g. 'cuda', 'cpu'
ENABLE_WANDB = False

AUDIT_EPISODES = 20
AUDIT_MAX_STEPS = 200
AUDIT_SEED = 1234
AUDIT_TIMEOUT_SEC = 10800

CONFIG_OVERRIDES = {
    'obs_mode': OBS_MODE,
    'encoder_name': ENCODER_NAME,
    'backbone_name': BACKBONE_NAME,
}
EXTRA_OVERRIDES = {}  # e.g. {'augment_data': True, 'dropout': 0.0}
CONFIG_OVERRIDES.update(EXTRA_OVERRIDES)

RUN_NAME = f"{BASE_RUN_NAME}__{EXPERIMENT_NAME}__manual__{datetime.now():%Y%m%d_%H%M%S}"
run_dir = PROJECT_ROOT / 'ckpt' / RUN_NAME
summary_path = run_dir / 'summary.json'
audit_report_path = run_dir / 'audit_report.json'
rollout_dir = PROJECT_ROOT / 'ckpt' / 'videos' / RUN_NAME

train_cmd = [
    sys.executable,
    '-u',
    str(TRIAL_SCRIPT),
    '--phase',
    'train-only',
    '--config',
    str(CONFIG_PATH),
    '--strategy',
    STRATEGY,
    '--run-name',
    RUN_NAME,
    '--experiment-name',
    EXPERIMENT_NAME,
    '--stage-epochs',
    str(STAGE_EPOCHS),
    '--checkpoint-every',
    str(CHECKPOINT_EVERY),
]
for key, value in CONFIG_OVERRIDES.items():
    train_cmd.extend(['--set', f'{key}={json.dumps(value)}'])
if DEVICE:
    train_cmd.extend(['--device', DEVICE])
train_cmd.append('--enable-wandb' if ENABLE_WANDB else '--no-enable-wandb')

audit_cmd = [
    sys.executable,
    '-u',
    str(TRIAL_SCRIPT),
    '--phase',
    'audit-only',
    '--run-dir',
    str(run_dir),
    '--eval-episodes',
    str(AUDIT_EPISODES),
    '--eval-seed',
    str(AUDIT_SEED),
    '--max-steps',
    str(AUDIT_MAX_STEPS),
    '--audit-timeout-sec',
    str(AUDIT_TIMEOUT_SEC),
    '--headless',
]
if DEVICE:
    audit_cmd.extend(['--device', DEVICE])

print('Train command:')
print(' '.join(train_cmd))
print()
print('Audit command:')
print(' '.join(audit_cmd))


In [ ]:
# Run training via the train-only CLI entry
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True, text=True)

if not summary_path.exists():
    raise FileNotFoundError(f'Expected summary file at {summary_path}, but it was not generated.')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary


## Notebook-safe RLBench workflow

- 训练阶段只跑 `train-only`。
- 成功率评估单独跑 `audit-only`。
- 要切 `pcd / rgb`、`encoder`、`backbone` 时，优先改上面的配置变量，而不是手改 import。


In [ ]:
# Optional: run standalone offline audit after training
RUN_AUDIT = False

print(' '.join(audit_cmd))
if RUN_AUDIT:
    subprocess.run(audit_cmd, cwd=PROJECT_ROOT, check=True, text=True)
    print(f'saved audit report -> {audit_report_path}')


In [ ]:
# Summarize the best checkpoint from audit_report.json
if not audit_report_path.exists():
    raise FileNotFoundError(
        f'No audit report found at {audit_report_path}. Run the audit-only cell first.'
    )

audit_report = json.loads(audit_report_path.read_text(encoding='utf-8'))
rows = [row for row in (audit_report.get('audit_records') or []) if row.get('success_rate') is not None]
if not rows:
    raise RuntimeError('No successful audit rows were found in audit_report.json.')

def _row_sort_key(row):
    epoch = row.get('epoch')
    epoch_sort = int(epoch) if epoch is not None else 10**9
    kind_sort = 0 if row.get('kind') == 'periodic' else 1
    return (-float(row['success_rate']), kind_sort, epoch_sort, str(row.get('label') or ''))

ranked_rows = sorted(rows, key=_row_sort_key)
best_row = ranked_rows[0]
periodic_rows = [row for row in ranked_rows if row.get('kind') == 'periodic']
best_periodic_row = None if not periodic_rows else periodic_rows[0]

best_summary = {
    'best_audited_ckpt_path': best_row['path'],
    'best_audited_label': best_row.get('label'),
    'best_audited_kind': best_row.get('kind'),
    'best_audited_epoch': best_row.get('epoch'),
    'best_audited_success_rate': best_row['success_rate'],
    'best_periodic_ckpt_path': None if best_periodic_row is None else best_periodic_row['path'],
    'best_periodic_epoch': None if best_periodic_row is None else best_periodic_row.get('epoch'),
    'best_periodic_success_rate': None if best_periodic_row is None else best_periodic_row['success_rate'],
    'top_rows': ranked_rows[:5],
    'artifact_manifest': str(PROJECT_ROOT / 'docs' / 'top10-checkpoint-manifest.json'),
}
best_summary


## Optional rollout video\n\n这一步只调用 `scripts/record_rollout_videos.py`。默认不执行，避免 notebook 里直接拉起 GUI。\n

In [ ]:
RUN_ROLLOUT = False
ROLLOUT_EPISODES = 1
ROLLOUT_CAMERA = 'front'
ROLLOUT_HEADLESS = True

render_ckpt_path = None
if audit_report_path.exists():
    audit_report = json.loads(audit_report_path.read_text(encoding='utf-8'))
    render_ckpt_path = audit_report.get('best_success_ckpt_path') or render_ckpt_path
if render_ckpt_path is None:
    render_ckpt_path = str(run_dir / 'epochs' / f'epoch_{STAGE_EPOCHS:04d}.pt')

rollout_cmd = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'scripts' / 'record_rollout_videos.py'),
    '--ckpt-path',
    str(render_ckpt_path),
    '--episodes',
    str(ROLLOUT_EPISODES),
    '--camera',
    ROLLOUT_CAMERA,
    '--output-dir',
    str(rollout_dir),
]
rollout_cmd.append('--headless' if ROLLOUT_HEADLESS else '--no-headless')
if DEVICE:
    rollout_cmd.extend(['--device', DEVICE])

print(' '.join(rollout_cmd))
if RUN_ROLLOUT:
    subprocess.run(rollout_cmd, cwd=PROJECT_ROOT, check=True, text=True)
